# Test Query Team Execution

This notebook demonstrates two ways to run the query team workflow:
1. **Using QueryManager**: Simulates the standard way of submitting a query and getting the final result.
2. **Using Direct Graph Stream**: Directly interacts with the LangGraph instance to observe intermediate steps.

## 七个模型综合总结与评分

本次评测考察了七个大型语言模型（A-G）在处理一系列关于基于指示剂置换分析（IDA）的奎宁电化学传感器的问答对时的表现。模型包括不同版本（gpt-4o, gpt-4.1, o4-mini, nano）以及不同运行模式（直接回答 vs. 查询增强后回答）。评测核心关注准确性、上下文理解、技术深度、结构和对目标受众（超分子化学研究者）的适切性。

**核心发现:**

1.  **IDA 术语理解是关键:** 模型的成败很大程度上取决于是否能正确理解核心术语 "IDA" 在此上下文（Indicator Displacement Assay）中的含义。只有模型 **A** 和 **C** 做到了这一点。模型 **B** 将其误解为碘乙酰胺/阻抗分析。模型 **D, E, F, G** 则一致将其误解为 Interdigitated Array (叉指阵列) 电极。这个基础性错误导致后者在关键问题（Q5, Q8）上的回答完全偏离主题，尽管它们在其他问题上可能表现出深度。
2.  **上下文感知能力差异悬殊:** 模型 **C** 在理解和利用问题背后隐含的特定传感器系统信息（β-CD 主体、具体性能指标等）方面表现突出，其答案与用户基准高度一致。模型 **A** 也展现了较好的上下文感知能力。其他模型则普遍缺乏这种能力，倾向于提供通用答案，或在错误的前提下进行推演。
3.  **查询增强 (`query + model`) 模式优势显著:** 对比 `query + nano` 模型（A, C）和 `direct answer` 模型（B, D, E, F, G），前者在理解术语、把握上下文方面表现明显更优。查询步骤似乎有效地引导模型聚焦于正确的领域和细节。
4.  **细节深度 vs. 核心准确性:** 模型 **F** 和 **G** (均为 o4-mini) 在提供极其详尽的技术细节、良好结构和广泛覆盖方面表现突出，甚至超过了 gpt-4.1 (D)。然而，它们在核心术语上的错误使得这些细节在关键问题上失去了意义，甚至可能产生误导。这表明，对于专业领域任务，核心概念的准确性优先于信息的堆砌。
5.  **模型大小与表现并非完全正相关:** 虽然 nano 模型 (B) 表现最差，但 o4-mini (F, G) 和 gpt-4.1 (D), gpt-4o (E) 这些更大的模型在关键的术语理解上同样失败。反而是在查询增强模式下，gpt-4.1 (C) 和 gpt-4o (A) (驱动 nano 回答) 表现更好。这凸显了运行模式和引导的重要性。

**各模型表现总结:**

* **模型 A (gpt-4o 查询 + nano):** 表现良好。正确理解 IDA，上下文感知较好，能提供具体数据。主要缺点是在 Q5 的结论（认为不存在此类传感器）与特定上下文相悖。
* **模型 B (nano 直接回答):** 表现最差。未能正确理解 IDA，回答普遍笼统、缺乏深度，是不可靠的选择。
* **模型 C (gpt-4.1 查询 + nano):** **表现最佳**。准确理解 IDA，上下文感知能力极强，答案精准且与用户基准高度吻合，完美契合任务需求。唯一的遗憾是缺失了 Q2 的回答。
* **模型 D (gpt-4.1 直接回答):** 表现不佳。虽然通用知识性问题回答结构尚可，并包含引用，但未能正确理解 IDA，且缺乏对特定上下文的把握。
* **模型 E (gpt-4o 直接回答):** 表现不佳。与 D 类似，未能正确理解 IDA，缺乏上下文感知，且细节和结构不如 D/F/G。
* **模型 F (o4-mini 直接回答):** 细节极其丰富，结构良好。但在关键的 IDA 理解上失败，且在 Q5 基于错误前提提供了看似详细实则误导的信息和引用。优点被核心错误掩盖。
* **模型 G (o4-mini 直接回答, high cost):** 细节极为丰富，结构好，通用知识问题回答出色。但在关键的 IDA 理解上同样失败。虽然 Q7 提供的“典型”数据非常接近真实值，显示出一定的潜力，但核心概念错误使其无法胜任此任务。

---

**最终评分与原因 (1-10分):**

1.  **模型 C (gpt-4.1 查询 + nano): 9.5 / 10**
    * **原因:** IDA 理解正确，上下文感知极佳，答案准确具体，与基准高度吻合。近乎完美，仅因缺失 Q2 略作扣分。
2.  **模型 A (gpt-4o 查询 + nano): 7.5 / 10**
    * **原因:** IDA 理解正确，上下文感知较好，提供了部分具体数据。但在关键问题 Q5 上结论与特定上下文冲突。
3.  **模型 G (o4-mini direct, high cost): 5.0 / 10**
    * **原因:** 通用知识问题极为详尽，结构好。但 IDA 理解错误（关键缺陷），导致核心问题回答错误，高成本未解决核心问题。细节丰富度使其略高于犯同样错误的 D/E/F。
4.  **模型 F (o4-mini direct): 4.5 / 10**
    * **原因:** 通用知识问题非常详尽，结构好。但 IDA 理解错误（关键缺陷），且 Q5 的误导性细节和引用问题严重。
5.  **模型 D (gpt-4.1 direct): 4.0 / 10**
    * **原因:** 结构和引用尚可。但 IDA 理解错误（关键缺陷），上下文感知差，未能提供特定数据。
6.  **模型 E (gpt-4o direct): 3.5 / 10**
    * **原因:** IDA 理解错误（关键缺陷），上下文感知差，细节和结构不如 D/F/G。
7.  **模型 B (nano direct): 2.0 / 10**
    * **原因:** IDA 理解错误（关键缺陷），回答普遍笼统，缺乏深度和准确性，基本不可用。

## Part 0: Setup (Imports and Ontology)

In [1]:
import sys
import os
sys.path.append(r"D:\\CursorProj\\Chem-Ontology-Constructor")
os.environ["PROJECT_ROOT"] = "D:\\\\CursorProj\\\\Chem-Ontology-Constructor\\\\"

from owlready2 import get_ontology
# 从 config.settings 导入 ONTOLOGY_SETTINGS 而不是 ONTOLOGY_CONFIG
from config.settings import ONTOLOGY_SETTINGS
# 本体现在在 ONTOLOGY_SETTINGS 初始化时加载，如果需要，可以通过 ONTOLOGY_SETTINGS.ontology 访问
# 例如: onto = ONTOLOGY_SETTINGS.ontology
# onto_additional = get_ontology("data/ontology/test.owl").load() # 可选的第二个本体

Setting owlready2.JAVA_EXE globally from settings.yaml: C:\Program Files\Java\jdk-23\bin\java.exe


In [36]:
from langchain_openai import ChatOpenAI

answer_llm = ChatOpenAI(
            model_name="gpt-4.1",
            temperature=0,
            max_tokens=10000,
        )

In [37]:
# Required Imports
import sys
import os
import json
import time
from typing import Dict, Any, List
from owlready2 import *
import asyncio # Needed for owlready2 async operations in some envs

# Import the OntologySettings class
from config.settings import OntologySettings # Keep ONTOLOGY_SETTINGS import for potential base_iri access

# Import necessary LLM and Query Team components
try:
    from autology_constructor.idea.query_team import QueryManager, Query, QueryStatus, create_query_graph
    from autology_constructor.idea.query_team.ontology_tools import OntologyTools
    from autology_constructor.idea.common.llm_provider import get_cached_default_llm
    print("Modules imported successfully.")
except ModuleNotFoundError as e:
    print(f"Error importing modules: {e}")
    print(f"Current sys.path: {sys.path}")

# Ensure LLM Provider is configured
try:
    llm = get_cached_default_llm()
    print(f"LLM: {llm.model_name}. \nAnsewr LLM: {answer_llm.model_name}")
    print("LLM Provider initialized successfully.")
except Exception as e:
    print(f"Error initializing LLM Provider: {e}\nPlease ensure API keys or necessary configurations are set.")
    llm = None

# --- Ontology Setup ---
print("Setting up test ontology using a new OntologySettings instance...")

# Define parameters for the new OntologySettings instance
# Assuming 'backup-2.owl' and 'backup-2-closed.owl' exist in 'data/ontology'
# Use the project root defined in the previous cell
project_root_path = os.environ.get("PROJECT_ROOT", ".")
ontology_dir = os.path.join(project_root_path, "data", "ontology")
# You might want to use the base_iri from the default settings or define a specific one for testing
test_base_iri = ONTOLOGY_SETTINGS.base_iri if 'ONTOLOGY_SETTINGS' in locals() else "http://www.test.org/chem_ontologies/backup-2"

try:
    # Instantiate OntologySettings directly
    # test_ontology_settings = ONTOLOGY_SETTINGS
    test_ontology_settings = OntologySettings(
        base_iri=test_base_iri,
        ontology_file_name="final.owl",  # Use the desired ontology file
        directory_path=ontology_dir,
        closed_ontology_file_name="IDA-closed.owl" # Adjust if your closed file has a different name pattern
    )
    # Access the loaded ontology via the instance's property
    test_onto = test_ontology_settings.ontology
    print(f"Successfully loaded ontology: {test_onto.base_iri}")
    print(f"From file: {test_ontology_settings.ontology_file_name} in {test_ontology_settings.directory_path}")

    # Optional: Print some details about the loaded ontology
    # print(f"Test Ontology '{test_onto.base_iri}' loaded with:")
    # print(f"- Classes ({len(list(test_onto.classes()))}): {[c.name for c in list(test_onto.classes())[:5]]}...") # Print first 5
    # print(f"- Individuals ({len(list(test_onto.individuals()))}): {[i.name for i in list(test_onto.individuals())[:5]]}...")
    # print(f"- Object Properties ({len(list(test_onto.object_properties()))}): {[p.name for p in list(test_onto.object_properties())[:5]]}...")
    # print(f"- Data Properties ({len(list(test_onto.data_properties()))}): {[p.name for p in list(test_onto.data_properties())[:5]]}...")

except Exception as e:
    print(f"Error creating OntologySettings or loading ontology 'backup-2.owl': {e}")
    print(f"Please ensure 'backup-2.owl' exists in '{ontology_dir}' and settings are correct.")
    test_onto = None # Set to None if loading failed

# # --- Old way (commented out) ---
# # print("Creating a simple in-memory ontology...")
# # # It's good practice to clear existing ontologies from the default world if running cells repeatedly
# # for o in list(default_world.ontologies.values()):
# #     if callable(getattr(o, '__destroy__', None)):
# #         try:
# #             destroy_entity(o)
# #         except Exception as destroy_err:
# #             print(f"Error destroying {o.base_iri}: {destroy_err}")
# #     else:
# #         print(f"Skipping destroy for non-callable __destroy__ or missing: {o.base_iri}")
# # test_onto_old = get_ontology("data/ontology/test.owl").load()
# # for o in list(default_world.ontologies.values()):
# #     print(f"has：{o.base_iri}")
# # print(f"Test Ontology '{test_onto_old.base_iri}' created with:\\n- Classes: {[c.name for c in test_onto_old.classes()]}\\n- Individuals: {[i.name for i in test_onto_old.individuals()]}\\n- Object Properties: {[p.name for p in test_onto_old.object_properties()]}\\n- Data Properties: {[p.name for p in test_onto_old.data_properties()]}\")

# Run async tasks if needed by owlready2 backend (usually not necessary for simple loading)
# try:
#     loop = asyncio.get_event_loop()
# except RuntimeError:
#     loop = asyncio.new_event_loop()
#     asyncio.set_event_loop(loop)
# loop.run_until_complete(asyncio.sleep(0)) # Run pending async tasks

Modules imported successfully.
LLM: gpt-4.1. 
Ansewr LLM: gpt-4.1
LLM Provider initialized successfully.
Setting up test ontology using a new OntologySettings instance...
Successfully loaded ontology: http://www.test.org/chem_ontologies/chem_ontology.owl#
From file: final.owl in D:\\CursorProj\\Chem-Ontology-Constructor\\data\ontology


## Part 1: Execution via QueryManager

In [4]:
qas = {
  "query_format_QA": [
    {
      "difficulty_level": 1,
      "query": "Quinine definition?",
      "answer": "Quinine is an alkaloid derived from cinchona tree bark, used historically for malaria and now as a bittering agent, but associated with adverse health effects like thrombocytopenia."
    },
    {
      "difficulty_level": 1,
      "query": "Indicator Displacement Assay (IDA) definition?",
      "answer": "An IDA is a sensing strategy based on host-guest recognition, often utilizing non-covalent interactions where an analyte displaces an indicator from a receptor, causing a detectable signal change (e.g., fluorescence or absorbance)."
    },
    {
      "difficulty_level": 2,
      "query": "What analyzes Quinine?",
      "answer": "Quinine is analyzed by techniques including Electrochemical Technique, High Performance Liquid Chromatography (HPLC), Colorimetric Assay, Fluorescence Assay, and High Resolution Mass Spectrometry (HRMS)."
    },
    {
      "difficulty_level": 2,
      "query": "List components of Indicator Displacement Assay (IDA).",
      "answer": "Components include beta-Cyclodextrin (beta-CD), Poly(N-acetylaniline), and Graphene."
    },
    {
      "difficulty_level": 3,
      "query": "Find electrochemical sensors based on Indicator Displacement Assay (IDA) used for Quinine detection.",
      "answer": "The Electrochemical Sensor constructed via IDA (Indicator Displacement Assay) has the detection target Quinine."
    },
    {
      "difficulty_level": 3,
      "query": "Which hosts use host-guest recognition and are integrated with electrochemical assays?",
      "answer": "Host-Guest Recognition is integrated with an Electrochemical Sensor. Beta-Cyclodextrin (β-CD) is a host involved in Host-Guest Recognition."
    },
    {
      "difficulty_level": 4,
      "query": "Compare the stability and reproducibility properties of the Electrochemical Sensor.",
      "answer": "The Electrochemical Sensor has high stability (acceptable peak current decrease within 21 days, 86.47% retained) and good reproducibility (RSD of 2.06% across seven electrodes)."
    },
    {
      "difficulty_level": 4,
      "query": "What techniques verify the Electrochemical Sensor based on IDA?",
      "answer": "The Electrochemical Sensor is verified by Differential Pulse Voltammetry (DPV), Cyclic Voltammetry (CV), Proton Nuclear Magnetic Resonance (H_NMR), Scanning Electron Microscopy (SEM), Electrochemical Impedance Analysis (EIS), and Fourier Transform Infrared (FTIR)."
    },
    {
      "difficulty_level": 5,
      "query": "Explain the sensing mechanism involving Methylene Blue (MB) displacement by Quinine from beta-Cyclodextrin (beta-CD).",
      "answer": "Methylene Blue (MB) forms an inclusion complex with beta-Cyclodextrin (beta-CD). Quinine, having a higher binding affinity, competitively displaces MB from the beta-CD cavity. This displacement causes a change in the electrochemical signal (e.g., DPV peak current) which is used for Quinine detection. Poly(N-acetylaniline) inhibits non-specific adsorption of MB, contributing to the assay's selectivity."
    },
    {
      "difficulty_level": 5,
      "query": "Summarize the role of Graphene in the described electrochemical sensor.",
      "answer": "Graphene (specifically reduced graphene oxide, rGO) is used as an electrode material in the sensor. It enhances electron transfer properties due to its superior electrical conductivity and large specific surface area, improving the sensor's performance. It serves as a platform onto which other components like Poly(N-acetylaniline) and beta-Cyclodextrin are deposited."
    }
  ],
  "question_format_QA": [
    {
      "difficulty_level": 1,
      "question": "Tell me about Quinine.",
      "answer": "Quinine is an alkaloid derived from cinchona tree bark, used historically for malaria and now as a bittering agent, but associated with adverse health effects like thrombocytopenia."
    },
    {
      "difficulty_level": 1,
      "question": "What is an Indicator Displacement Assay?",
      "answer": "An IDA is a sensing strategy based on host-guest recognition, often utilizing non-covalent interactions where an analyte displaces an indicator from a receptor, causing a detectable signal change (e.g., fluorescence or absorbance)."
    },
    {
      "difficulty_level": 2,
      "question": "What techniques are used to analyze Quinine?",
      "answer": "Quinine is analyzed by techniques including Electrochemical Technique, High Performance Liquid Chromatography (HPLC), Colorimetric Assay, Fluorescence Assay, and High Resolution Mass Spectrometry (HRMS)."
    },
    {
      "difficulty_level": 2,
      "question": "What are the components of an Indicator Displacement Assay?",
      "answer": "Components include beta-Cyclodextrin (beta-CD), Poly(N-acetylaniline), and Graphene."
    },
    {
      "difficulty_level": 3,
      "question": "Are there electrochemical sensors using Indicator Displacement Assay (IDA) to detect Quinine?",
      "answer": "The Electrochemical Sensor constructed via IDA (Indicator Displacement Assay) has the detection target Quinine."
    },
    {
      "difficulty_level": 3,
      "question": "Which host molecules use host-guest recognition in electrochemical assays?",
      "answer": "Host-Guest Recognition is integrated with an Electrochemical Sensor. Beta-Cyclodextrin (β-CD) is a host involved in Host-Guest Recognition."
    },
    {
      "difficulty_level": 4,
      "question": "How stable and reproducible is the electrochemical sensor that uses an Indicator Displacement Assay (IDA) for detecting Quinine?",
      "answer": "The Electrochemical Sensor has high stability (acceptable peak current decrease within 21 days, 86.47% retained) and good reproducibility (RSD of 2.06% across seven electrodes)."
    },
    {
      "difficulty_level": 4,
      "question": "How is the electrochemical sensor that uses an Indicator Displacement Assay (IDA) for detecting Quinine verified?",
      "answer": "The Electrochemical Sensor is verified by Differential Pulse Voltammetry (DPV), Cyclic Voltammetry (CV), Proton Nuclear Magnetic Resonance (H_NMR), Scanning Electron Microscopy (SEM), Electrochemical Impedance Analysis (EIS), and Fourier Transform Infrared (FTIR)."
    },
    {
      "difficulty_level": 5,
      "question": "In the electrochemical sensor that uses an Indicator Displacement Assay (IDA) for detecting Quinine, how does Quinine displace Methylene Blue from beta-Cyclodextrin?",
      "answer": "Methylene Blue (MB) forms an inclusion complex with beta-Cyclodextrin (beta-CD). Quinine, having a higher binding affinity, competitively displaces MB from the beta-CD cavity. This displacement causes a change in the electrochemical signal (e.g., DPV peak current) which is used for Quinine detection. Poly(N-acetylaniline) inhibits non-specific adsorption of MB, contributing to the assay's selectivity."
    },
    {
      "difficulty_level": 5,
      "question": "What does Graphene do in the electrochemical sensor that uses an Indicator Displacement Assay (IDA) for detecting Quinine?",
      "answer": "Graphene (specifically reduced graphene oxide, rGO) is used as an electrode material in the sensor. It enhances electron transfer properties due to its superior electrical conductivity and large specific surface area, improving the sensor's performance. It serves as a platform onto which other components like Poly(N-acetylaniline) and beta-Cyclodextrin are deposited."
    }
  ]
}

In [5]:
new_qas = [
  {
    "question": "What is a cryptand?",
    "query": "Retrieve the definition or description associated with the chemical class 'Cryptand'. Look for annotations like rdfs:comment, skos:definition, or specific meta-properties on the 'Cryptand' class definition.",
    "answer": "A cryptand is a type of macrocyclic ligand, specifically a supramolecular host, featuring a three-dimensional cavity that allows it to form stable complexes by encapsulating guest ions or molecules.",
    "difficulty_level": 1
  },
  {
    "question": "Is pyrrole considered an aromatic system?",
    "query": "Verify if an `rdfs:subClassOf` axiom exists where 'Pyrrole' is declared as a subclass of the 'AromaticSystem' class within the ontology.",
    "answer": "Yes, pyrrole is classified as an aromatic system.",
    "difficulty_level": 1
  },
  {
    "question": "What types of molecules typically act as guests for supramolecular hosts?",
    "query": "Identify the `rdfs:range` axiom defined for the object property 'binds_guest'. This indicates the general class of entities that can be bound by entities that are in the domain of 'binds_guest' (typically 'SupramolecularHost').",
    "answer": "Typically, molecules classified as 'GuestMolecule' (which can include anions, cations, or neutral molecules) act as guests for supramolecular hosts.",
    "difficulty_level": 2
  },
  {
    "question": "What are some specific types of macrocycles?",
    "query": "List all `owl:Class` entities for which an `rdfs:subClassOf` axiom exists, explicitly stating 'Macrocycle' as the direct parent class.",
    "answer": "Specific types of macrocycles include cryptands, calixarenes, pillararenes, and cyclodextrins.",
    "difficulty_level": 2
  },
  {
    "question": "When a calixarene containing pyrrole groups binds an anion, what specific non-covalent interactions are typically involved?",
    "query": "Analyze OWL axioms or property restrictions associated with the 'Calixarene' class, particularly when it 'has_functional_group' 'Pyrrole' and 'binds_guest' an 'Anion'. Identify the specific subclasses of 'InteractionType' (e.g., 'HydrogenBond', 'AnionPiInteraction') that are linked via 'interacts_via' in these defined scenarios.",
    "answer": "When a calixarene containing pyrrole groups binds an anion, non-covalent interactions such as HydrogenBond (often from the pyrrole NH) and AnionPiInteraction (between the anion and the pyrrole ring) are typically involved.",
    "difficulty_level": 3
  },
  {
    "question": "Are there known supramolecular hosts that are derivatives of calixarenes and also feature pyrrole functional groups?",
    "query": "Search for 'SupramolecularHost' classes or typical instances that are described as being 'is_derivative_of' the 'Calixarene' class AND are also associated with the 'Pyrrole' class via the 'has_functional_group' property.",
    "answer": "Yes, supramolecular hosts that are derivatives of calixarenes and feature pyrrole functional groups are known, a prominent example being calix[n]pyrroles.",
    "difficulty_level": 3
  },
  {
    "question": "What are common applications for cage molecules compared to macrocycles, and do they have any overlapping uses?",
    "query": "1. Identify 'Application' subclasses (e.g., 'Sensing', 'Catalysis', 'DrugDelivery', 'Separation') linked to the 'CageMolecule' class via the 'has_application' property (possibly through class axioms or restrictions). 2. Perform the same analysis for the 'Macrocycle' class. 3. Compare these sets of 'Application' subclasses to identify distinct and common areas.",
    "answer": "Cage molecules are commonly utilized in applications like catalysis and chemical separation. Macrocycles frequently find use in areas such as molecular sensing and drug delivery. An example of an overlapping application could be sensing, where both classes of compounds might be employed.",
    "difficulty_level": 4
  },
  {
    "question": "What types of supramolecular hosts are known to bind anions primarily through anion-π interactions?",
    "query": "Find 'SupramolecularHost' subclasses or characteristic descriptions where OWL axioms (e.g., `owl:equivalentClass` or `rdfs:subClassOf` involving restrictions on 'binds_guest' with 'Anion', and 'interacts_via' with 'AnionPiInteraction') define this specific binding mode.",
    "answer": "Supramolecular hosts that possess electron-deficient aromatic systems, such as certain calixarene derivatives (like calix[n]pyrroles) or other specifically designed π-acidic macrocycles, are known to bind anions primarily through AnionPiInteraction.",
    "difficulty_level": 4
  },
  {
    "question": "Why are supramolecular hosts containing pyrrole units generally effective at binding anions?",
    "query": "Analyze the defined properties of the 'Pyrrole' class (e.g., its classification as an 'AromaticSystem', its NH group) and its typical involvement in 'InteractionType' classes like 'HydrogenBond' and 'AnionPiInteraction' when 'Pyrrole' is a 'FunctionalGroup' of a 'SupramolecularHost' binding an 'Anion'. Synthesize an explanation for this effectiveness based on these defined chemical characteristics and interaction capabilities.",
    "answer": "Supramolecular hosts with pyrrole units are effective for anion binding due to two main features of pyrrole: its NH group can act as a hydrogen bond donor, forming HydrogenBond interactions with anions, and its electron-deficient aromatic π-system can engage in favorable AnionPiInteractions with anions. This combination enhances binding affinity and selectivity.",
    "difficulty_level": 5
  },
  {
    "question": "What is the role of non-covalent interactions like hydrogen bonds and anion-π interactions in the formation of supramolecular host-guest complexes?",
    "query": "Examine how 'InteractionType' subclasses (e.g., 'HydrogenBond', 'AnionPiInteraction') are axiomatically linked via the 'interacts_via' property in scenarios where 'SupramolecularHost' classes bind 'GuestMolecule' classes (often described by 'binds_guest' relationships). Summarize their function based on these defined ontological roles.",
    "answer": "Non-covalent interactions, such as hydrogen bonds and anion-π interactions, are crucial for molecular recognition between hosts and guests. They act as the primary driving forces that determine the stability, selectivity, and overall formation of supramolecular host-guest complexes.",
    "difficulty_level": 5
  }
]
 

In [6]:
IDA_qas = [
    {
    "question": "Regarding the system described in the paragraph below, please first attempt to verify if it is a documented Indicator Displacement Assay (IDA) system. If it cannot be directly confirmed, then provide an assessment of its potential to function as an IDA system based on the given details.\nThis chemical system utilizes WP6, a water-soluble carboxylato pillar[6]arene, as the synthetic receptor, belonging to the pillararene macrocycle family. Safranine T (ST), a phenazine-based dye, serves as the fluorophore indicator. Caffeine, a xanthine alkaloid, is the selected analyte competitor. This setup suggests a host-guest system designed for molecular recognition of Caffeine by the WP6 receptor, with binding events monitored through changes in the fluorescence of Safranine T."
  },
  {
    "question": "Regarding the system described in the paragraph below, please first attempt to verify if it is a documented Indicator Displacement Assay (IDA) system. If it cannot be directly confirmed, then provide an assessment of its potential to function as an IDA system based on the given details.\nThe image displays a supramolecular assembly featuring β-cyclodextrin (β-CD), a cyclic oligosaccharide, as the synthetic receptor. Methylene Blue (MB), a well-known phenothiazine dye, acts as the fluorophore indicator. Quinine, a quinoline-derived alkaloid, is presented as the selected analyte competitor. This system is characteristic of host-guest chemistry aimed at studying the interaction between β-cyclodextrin and Quinine, where Methylene Blue signals the binding or displacement."
  },
  {
    "question": "Regarding the system described in the paragraph below, please first attempt to verify if it is a documented Indicator Displacement Assay (IDA) system. If it cannot be directly confirmed, then provide an assessment of its potential to function as an IDA system based on the given details.\nThis chemical architecture involves a synthetic receptor TCC, which is a resorcinarene-based cavitand functionalized with four imidazole-acetic acid sodium salt arms. The fluorophore indicator is DSMI, a styryl-pyridinium dye. The selected analyte competitor, labeled 'Choline(Cho)' but structurally depicted as Acetylcholine, is a quaternary ammonium ester. This system is likely designed for the molecular recognition of Acetylcholine by the TCC receptor, with interactions reported by the DSMI fluorescent probe."
  },
  {
    "question": "Regarding the system described in the paragraph below, please first attempt to verify if it is a documented Indicator Displacement Assay (IDA) system. If it cannot be directly confirmed, then provide an assessment of its potential to function as an IDA system based on the given details.\nThe synthetic receptor in this system is TCC, specified as a macrocyclic cavitand with R1 groups being CH2CO2Na (carboxymethyl sodium salt) and R2 groups being Et (ethyl) on its benzimidazole units. DTMI, a cyanine-type styryl benzothiazole dye (with iodide as counterion), functions as the fluorophore indicator. Butyrylcholine (Bucho), an ester of choline, serves as the selected analyte competitor. This assembly is designed for investigating the host-guest interactions between the specific TCC receptor and Butyrylcholine, using the DTMI dye to signal these events."
  },
  {
    "question": "Regarding the system described in the paragraph below, please first attempt to verify if it is a documented Indicator Displacement Assay (IDA) system. If it cannot be directly confirmed, then provide an assessment of its potential to function as an IDA system based on the given details.\nThis chemical system features Cucurbit[8]uril, abbreviated as CB[8], acting as a synthetic receptor from the cucurbituril family. The system employs Proflavine, or PF, an acridine dye, as the fluorophore indicator. Gefitinib, labeled GEF, an anilinoquinazoline compound and EGFR inhibitor, is the selected analyte competitor. This assembly is characteristic of a supramolecular host-guest system designed for molecular recognition studies, where CB[8] interacts with Gefitinib, and this interaction is potentially signaled by changes in the fluorescence of Proflavine, possibly through competitive binding or allosteric effects."
  },
  {
    "question": "Regarding the system described in the paragraph below, please first attempt to verify if it is a documented Indicator Displacement Assay (IDA) system. If it cannot be directly confirmed, then provide an assessment of its potential to function as an IDA system based on the given details.\nThis image depicts a supramolecular assembly involving Cucurbit[8]uril (CB[8]) as a synthetic receptor, which is a macrocyclic host compound. Methylene Blue (MB), a phenothiazine dye, functions as the fluorophore indicator. Amantadine (AMA), an antiviral drug with a tricyclic adamantane structure, serves as the selected analyte competitor. The system suggests a host-guest chemistry approach for the potential detection or binding study of Amantadine with Cucurbit[8]uril, utilizing the signaling properties of Methylene Blue."
  },
  {
    "question": "Regarding the system described in the paragraph below, please first attempt to verify if it is a documented Indicator Displacement Assay (IDA) system. If it cannot be directly confirmed, then provide an assessment of its potential to function as an IDA system based on the given details.\nThe chemical architecture shown consists of p-sulfonatocalix[4]arene, or SCX4, a macrocyclic compound from the calixarene family, acting as the synthetic receptor. An Acridine dye, shown in its protonated form (AcH$^+$), is utilized as the fluorophore indicator. Acetylcholine (AcCh), a neurotransmitter, is presented as the selected analyte competitor. This setup describes a potential sensing ensemble where the p-sulfonatocalix[4]arene receptor binds Acetylcholine, and this molecular recognition event is reported by changes in the fluorescence characteristics of the Acridine dye."
  },
]

In [7]:
hard_qas = [
  {
    "question": "Explain the interplay of enthalpy and entropy in the binding of anions to calixarene hosts, considering how structural variations in the host might influence these thermodynamic contributions.",
    "answer": "For sulfonatocalix1arene (SC4) binding amino acids like Lys and Arg (which have cationic/anionic components), ITC studies showed the binding is primarily enthalpically driven (ΔH = -14.4 kJ/mol for Lys, -20.3 kJ/mol for Arg) with only slight differences in entropy (ΔS = 2.0 kJ/mol for Lys, -2.1 kJ/mol for Arg). For SC6 (sulfonatocalix3arene), compared to SC4, studies showed higher affinities and selectivities, attributed to its larger and more flexible cavity. While not specifically calixarenes, ITC studies on iodotriazole (1a) and methyltellanyl triazole (2Te) foldamer receptors binding iodide (I-) in water showed negative enthalpy contributions (ΔH = -23.2 kJ/mol) and small positive entropy contributions (TΔS = 2.7 kJ/mol) for the 1:1 binding step. The HB analog (1aH) binding I- also showed negative enthalpy and positive entropy contributions. The hydrophobic character of σ-hole donor atoms like I and Te is expected to augment the hydrophobicity of the backbone and favor aggregation in water. The dimeric capsules of these receptors partially shelter the hosted anion from solvent molecules, with an average of 2 water molecules surrounding the encapsulated iodide, compared to approximately 20 for free iodide. The flexibility of PEG side chains acting as lids can limit water access to the binding cavity."
  },
  {
    "question": "How does the size and flexibility of the calixarene cavity, specifically comparing SC4, SC6, and SC8, influence their binding affinity and selectivity towards amino acids like Lys and Arg?",
    "answer": "For sulfonatocalix1arene (SC4), binding studies with amino acids (Lys, Arg) indicate that the terminal amino/guanidino groups of the side chain are included in the cavity, while the α-amino group is repelled by the carboxylate group. Arg shows a higher affinity towards SC4 compared to Lys. Crystallographic structures support recognition through inclusion of side chains in the SC4 cavity. Studies comparing SC4 and SC6 showed that SC6 has higher affinities and selectivities for different amino acids than SC4. This difference is attributed to the larger and more flexible cavity of SC6 compared to SC4. The sources mention SC4 and SC6 in the context of amino acid binding but do not provide information on SC8's interaction with amino acids."
  },
  {
    "question": "Describe how differences in guest structure, including charge number, π-conjugation, and size, affect the binding strength and mode of interaction with resorcinarene hosts like SR4A5.",
    "answer": "Source shows that SR4A5 interacts with guests G7 and G8 in water, as indicated by changes in UV/vis absorption spectra and visible color changes upon mixing the host and guests. The provided sources do not contain specific details on how variations in guest structure characteristics such as charge number, degree of π-conjugation, or size systematically affect the binding strength and mode of interaction with SR4A5."
  },
  {
    "question": "Based on observed fluorescence phenomena like quenching, enhancement, and blue shifts upon guest binding to indicator-host complexes, how can these changes be mechanistically linked to the binding event?",
    "answer": "Fluorescence changes upon guest binding indicate that the fluorophore's environment or electronic state is altered by the host-guest interaction. Quenching: In the case of the PAL-Q11 complex and phenylalanine (Phe), the fluorescence of the PAL-Q11 complex is dramatically quenched by Phe. This is explained by Phe displacing PAL from the Q11 cavity, suggesting that the encapsulated state provides fluorescence, while the free state or a different complexed state does not, leading to a \"turn-off\" detection method. Similarly, fluorescence quenching of pyrene (G2) upon addition of triazinophanes (1a-c) was observed, indicating complex formation. Fluorescence quenching of an indicator dye displaced by a guest can signal binding. Enhancement: Binding of 7-DCCAE to CB7 and CB8 resulted in modulation of its photophysical properties, observed through fluorescence emission spectroscopy. An unsymmetrical cyclophane receptor (7) with pyrene and naphthalimide showed fluorescence enhancement (sixfold for ATP, fivefold for CTP) upon binding nucleotides in aqueous solution. Quantum chemical calculations suggest that hydrogen bonding interactions between the nucleobases (adenine, cytosine) and amine linkers of the host stabilize the complex, likely influencing the electronic or conformational state of the fluorophores. Blue/Red Shifts: Different guest/host stoichiometries and assembly modes can lead to tunable fluorescent signals, including red-shifted emission. For OPV2+-H (5C) forming a 2:3 assembly with Q20, a red-shifted emission was observed. This is attributed to the assembly mode strengthening the acceptor-donor-acceptor (A-D-A) electronic interaction, increasing the π-surface, and enhancing the electron-donor group, promoting the ICT (intramolecular charge transfer) effect of the guest. A blue shift was observed for receptor 7 upon protonation in water, leading to a non-stacked conformation, which is opposite to the FRET observed in organic solvent where it is in a stacked conformation. Lifetime Changes: Changes in fluorescence lifetime upon host addition indicate complexation. For NBA with CB7, a longer lifetime component appeared upon host addition, attributed to the complex formation. This suggests the dye is protected from solvent interactions (like proton transfer in water), leading to a longer excited-state lifetime. Mechanism: The changes can be linked to the guest altering the fluorophore's local environment (e.g., moving it from polar water to a hydrophobic cavity, affecting interactions with solvent like proton transfer), inducing conformational changes in the host or guest, altering electronic interactions within the complex, or leading to displacement of an indicator dye."
  },
  {
    "question": "Discuss how the position and magnitude of complex-induced chemical shifts (Δδ) in NMR spectroscopy provide insights into the specific inclusion geometry and depth of a guest molecule within a host cavity.",
    "answer": "Complexation between a host and guest often leads to changes in the chemical shifts of their nuclei in NMR spectra. Upfield shifts (shielding) are commonly observed for guest protons included within the shielding cone of the host's aromatic or macrocyclic cavity. The magnitude of these complex-induced chemical shifts (Δδ or CIS) is directly related to the binding geometry. For example, when Nε-Me-Lys·HCl binds to host H3 in water, the significant upfield shifts (-3.3 ppm for the N-Me group and -0.97 ppm for the α methylene) suggest inclusion of the terminal N-Me group within the host's cavity. Similarly, significant upfield shifts for amino acid protons bound to SC4 suggest inclusion of their side chains in the SC4 cavity. The specific protons experiencing shifts indicate which parts of the guest molecule are interacting with or are located within the host's shielding region, providing information about the inclusion geometry. For instance, in the Q20-L-Trp/L-Phe systems, 1H NMR confirmed that the included AC molecule was replaced by L-Trp or L-Phe within the Q20 cavity. NMR can also reveal conformational details of the complex, such as a U-shaped conformation upon inclusion in a macrocycle cavity. Two-dimensional NMR techniques like ROESY can show through-space correlations between host and guest protons, confirming inclusion and providing spatial relationships."
  },
  {
    "question": "Explain the factors that allow cucurbiturils like CB7 and CB8 to preferentially recognize aromatic amino acids, and how their differing cavity sizes lead to variations in stoichiometry (1:1 vs 1:2) and selectivity among Phe, Trp, and Tyr.",
    "answer": "Cucurbiturils, such as CB7 and CB8, are known to recognize aromatic amino acids. Studies indicate that the binding of CB11 to phenylalanine (Phe) is strong compared to other amino acids. Cavity Size: CB7 has a smaller cavity than CB8. CB7 is noted to form a 1:1 complex with Phe. CB8, with its larger cavity, can accommodate two guest molecules, leading to 1:2 stoichiometry, or bind amino acids in the presence of auxiliary guests to form heteroternary complexes. Stoichiometry: CB8 can form a homoternary complex with two Phe molecules, resulting in a 1:2 host-guest inclusion complex. A crystal structure confirmed two L-Phe molecules within the Q20 cavity in a 1:2 stoichiometry. CB8 can also form 1:1:1 heteroternary complexes, for example, with Trp and methyl viologen (MV). ITC analyses of peptide sequences containing Phe, Trp, or Tyr binding to MV·CB20 showed a 1:1 molar ratio for the heteroternary complex. Binding Modes/Interactions: For Q20-L-Phe, the 1:2 complex in the crystal structure shows the two L-Phe molecules stabilized by π-π interactions between their aromatic rings. Hydrogen bonds between the amino acid and the carbonyl oxygens of Q20 portals also contribute to stability. For inclusion complexes, the aromatic side chain of the amino acid can be completely embedded inside the hydrophobic cavity, leading to the release of water molecules. This configuration change can significantly impact binding energy. Selectivity: CB7 preferentially recognizes aromatic amino acids. For CB8, the binding of peptide sequences containing Phe, Trp, or Tyr in the context of a preformed MV·CB20 complex shows selectivity dependent on the specific peptide sequence and the orientation of the aromatic side chain. Replacing Phe with Ala in a specific motif eliminated binding. A different Phe-based epitope also showed no binding to MV·CB20 despite the Phe residue being accessible, suggesting a fundamental requirement to maintain a favorable amino acid side chain orientation within the epitope for high affinity heteroternary complex formation. Molecular dynamics simulations showed that for tripeptides with N-terminal Phe (FGX), binding affinity to CB11 can reach nanomolar levels when X is Glu, Lys, or Arg, and this affinity is significantly changed by reversing the sequence or inserting a Gly, indicating sequence selectivity beyond just the presence of Phe."
  },
  {
    "question": "How can synthetic receptors achieve discrimination between different methylation states of lysine residues on peptide tails?",
    "answer": "Synthetic receptors can be designed to recognize and discriminate between post-translationally methylated lysines. One approach involves using a receptor unit that specifically complexes a particular methylation state of lysine. For example, trisulfonatocalixarene 21 was used for the affinity labeling of peptides containing trimethylated lysine (Kme3). The receptor unit of molecule 21 specifically complexes the trimethyllysine in the peptide guest. The design can incorporate functionalities that exploit differences in charge, size, or hydrogen bonding ability among different methylation states (e.g., Lys, Kme1, Kme2, Kme3) to achieve selective binding. Source focuses on recognition of Kme3 by a specific calixarene-based receptor unit."
  },
  {
    "question": "Describe the role of pH in modulating the binding interactions between hosts like SCX4 or calixarenes and charged guests.",
    "answer": "pH can modulate binding interactions, particularly with charged guests and hosts that have ionizable groups. For the SCX4 host interacting with acetylcholine, pH affects the prototropic equilibrium between the protonated (AcH+) and neutral (Ac) forms of acetylcholine. A thermodynamic model for the system includes pKa values for the free and bound forms of the guest, indicating that the distribution of guest species and their binding to the host is pH-dependent. Calixarenes, such as sulfonatocalix1arene (SC4), bear charged groups (sulfonates) and are studied under specific pH conditions (e.g., pH 8 or pH 5) when binding charged amino acids, highlighting pH as a relevant factor for these interactions. The ionization state of both the host and the guest, which is controlled by pH, influences electrostatic interactions, a significant driving force in binding charged species. More generally, the fluorescence properties of some receptors are pH-dependent, reflecting changes in protonation state that can affect conformation and binding ability."
  },
  {
    "question": "Synthesize the various experimental techniques described for characterizing host-guest complexes and determining binding affinities (e.g., NMR, fluorescence, ITC, UV-Vis, SPR), explaining for each method what type of molecular information it primarily provides and inferring from the contexts in which they are used what makes them suitable for studying different host-guest systems.",
    "answer": "NMR (1H, 2D, etc.): Provides information about the structure, conformation, and dynamics of molecules and complexes in solution. Complex-induced chemical shifts (CIS) indicate which parts of the guest are included in the host cavity and provide insights into the binding geometry and depth. Changes in signal multiplicity or appearance (e.g., doublets becoming merged) can indicate guest displacement. 2D techniques like ROESY can confirm spatial proximity between host and guest protons within the complex. Suitable for systems where binding causes detectable changes in the local magnetic environment of nuclei. Fluorescence Spectroscopy (Steady State and Time-Resolved): Detects changes in fluorescence properties (intensity, emission wavelength, lifetime) upon complex formation. Intensity changes (quenching or enhancement) can signal binding events and are used for sensing and detection. Emission wavelength shifts (blue or red shifts) suggest changes in the fluorophore's environment or electronic state. Fluorescence lifetime measurements indicate how complexation affects the excited-state dynamics and protection from solvent. Suitable for systems involving fluorescent hosts or guests, or those allowing indicator displacement assays with fluorescent dyes. Allows calculation of binding constants from titration data. ITC (Isothermal Titration Calorimetry): Directly measures the heat change associated with the binding process. Provides thermodynamic parameters (Ka, ΔH, ΔS) in a single experiment, revealing the driving forces (enthalpic vs. entropic) for complexation. Also determines the stoichiometry of the complex. Suitable for systems with detectable heat changes upon binding; provides comprehensive thermodynamic data. UV-Vis Spectroscopy: Monitors changes in the absorption spectrum of host, guest, or complex upon mixing. Indicates complex formation if one or both components have UV-Vis absorption. Can be used to calculate binding constants in some cases. Suitable for systems where binding affects the ground-state electronic transitions of chromophores. SPR (Surface Plasmon Resonance) or Nanomechanical Sensing: Used to measure binding events occurring at a surface, providing binding isotherms. Suitable for studying interactions with surface-immobilized components and can be used for real sample analysis due to robustness. Other Techniques: Crystallography provides the detailed 3D solid-state structure of host-guest complexes, showing precise interaction geometries, distances, and conformations. Molecular Dynamics Simulations offer theoretical insights into binding modes, dynamics, water effects, and conformational changes in solution. Quantum Chemical Calculations (DFT) optimize molecular structures and energies of complexes and help understand specific interactions like hydrogen bonding or electrostatic potentials. Mass Spectrometry (ESI-MS) can confirm the formation and stoichiometry of complexes. CD (Circular Dichroism) is used to study chiral interactions and conformations. The choice of technique depends on the system's properties (e.g., fluorescence, UV-Vis activity), solubility, whether solid-state structure is required, and the specific information sought (thermodynamics, structure, dynamics)."
  },
  {
    "question": "Based on the principles of molecular recognition evident from the described host-guest systems (e.g., electrostatic interactions, hydrophobic effects, cavity size complementarity, π-π interactions, hydrogen bonding, conformational changes), propose a conceptual strategy for designing a synthetic receptor capable of selectively binding a small, amphiphilic molecule possessing both positively charged and aromatic functionalities in an aqueous environment.",
    "answer": "Strategy: Design a water-soluble synthetic receptor that incorporates specific binding sites tailored to interact simultaneously with both the positively charged and aromatic parts of the amphiphilic guest molecule. The receptor should have a hydrophilic exterior to ensure solubility in water while providing a hydrophobic internal environment conducive to binding. Components & Interactions: Hydrophobic Cavity/Binding Site: To interact with the aromatic functionality, the receptor should have a hydrophobic cavity or a defined hydrophobic region. The size and shape of this cavity should be complementary to the aromatic ring of the guest to enhance binding affinity and selectivity through size complementarity. Macrocyclic hosts like calixarenes, pillararenes, cucurbiturils, or cavitands are examples that provide hydrophobic cavities. Pi-pi interactions between aromatic units in the host (if present) and the guest can further stabilize the complex. The expulsion of high-energy water molecules from this hydrophobic cavity upon guest inclusion will provide a favorable entropic contribution to binding. Electrostatic Interaction Site: To interact with the positively charged functionality of the guest, the receptor could incorporate anionic groups or electron-rich aromatic surfaces (cation-π interactions). Anionic groups (e.g., sulfonates, carboxylates) could be strategically placed near the opening or within the binding site to engage the positive charge via attractive electrostatic forces. Calixarenes with sulfonate or carboxylate groups serve as examples of hosts utilizing anionic charges for recognition. The contribution of this electrostatic interaction is significant. Water Solubility: The receptor molecule should be functionalized with sufficient hydrophilic groups (e.g., sulfonate, carboxylate, poly(ethylene glycol) chains, pyridinium ions on the exterior) to ensure good solubility in water, which is crucial for recognition in biological or environmental matrices. Structural Integrity/Preorganization: The receptor structure could be somewhat preorganized (e.g., a foldamer, a cavitand) to minimize the entropic cost of complex formation and ensure that the hydrophobic cavity and electrostatic interaction site are correctly positioned relative to each other to match the geometry of the guest molecule. However, some flexibility can also be beneficial, allowing for induced fit. Hydrogen bonding interactions within the receptor or between the receptor and guest can help stabilize preferred conformations. Selectivity: Fine-tuning the size, shape, and chemical nature (hydrophobicity, charge distribution, hydrogen bonding potential) of the binding sites is key for achieving selectivity for the target amphiphilic molecule over similar guests. This involves balancing multiple non-covalent interactions. For example, matching the size of the hydrophobic cavity to the aromatic ring is critical. The specific arrangement and distance between the charged site and the hydrophobic site should be optimized for the guest's structure. This strategy combines principles of hydrophobic encapsulation for the aromatic part and electrostatic attraction for the charged part within a water-soluble framework, guided by insights from various host-guest systems described in the sources."
  }
]

In [17]:
new_hard_qas = [
    {
      "question": "For a hydrophilic cavitand host, describe the experimental technique used to study its complex formation with ω-fatty acid guests and identify one specific ω-6 fatty acid guest whose interaction was investigated using this method, referencing observed spectral changes."
    },
    {
      "question": "Based on reported dissociation constants for a receptor interacting with peptide conjugates, analyze how increasing the valency of the amide conjugate (from monovalent to divalent and trivalent) affects binding affinity towards a specific histone tail, providing specific dissociation constant values for each valency."
    },
    {
      "question": "When using a specific cucurbituril as a host in aqueous solution, compare the experimentally determined stability constants for its complexes with Phenylalanine and Lysine, noting which amino acid exhibits significantly higher affinity based on the quantitative data provided."
    },
    {
      "question": "Detail the types of electrochemical characterization performed to assess stepwise assembly processes on a glassy carbon electrode, specifically mentioning the observed differences in impedance or potential profiles for a reduced graphene oxide modified electrode compared to a graphene oxide modified electrode."
    },
    {
      "question": "Describe the deduced binding modes for singly charged π-aromatic guests with a resorcinarene based on NMR data, explaining how the inclusion geometry differs between these two types of guests and the resorcinarene cavity."
    },
    {
      "question": "Considering the calculated thermodynamic parameters for adsorption of a specific fluorouracil derivative on calixarene structures, identify which of the studied complexes exhibits the most favorable adsorption free energy and state its corresponding enthalpy value."
    },
    {
      "question": "Discuss the binding interaction between a specific host and anthracene derivatives based on fluorescence titration studies, explaining the difference in binding affinity observed for a dicationic guest compared to a monocationic guest."
    },
    {
      "question": "Explain the effect of substituting alanine for phenylalanine in a specific loop of a protein domain on its interaction with a methyl viologen-cucurbituril complex, referencing the experimental technique used to characterize this binding."
    },
    {
      "question": "For the molecular recognition of tripeptides containing N-terminal phenylalanine by a specific cucurbituril, state the binding free energy and association constant for the most prominent tripeptide identified as having nanomolar binding affinity."
    },
    {
      "question": "Identify the multiple distinct guest/host stoichiometric ratios reported for dynamic host-guest assemblies formed between specific oligophenylenevinylene derivatives and a macrocycle."
    }
  ]


In [29]:
num = 10
# 定义新的十个查询
queries = [item["question"] for item in qas["question_format_QA"][:num]]

revised_queries = [item["question"] for item in qas["question_format_QA"][:num]]

# 为所有查询定义统一的上下文
query_context = {
    "ontology": test_ontology_settings,
    "originating_team": "test_notebook",
    "originating_stage": "manual_test",
    "query_type": "information_retrieval" # 对所有查询使用信息检索类型
}

print(len(queries),len(revised_queries))

10 10


In [45]:
num = len(new_qas)

queries = [item["question"] for item in new_qas[:num]]

revised_queries = [item["question"] for item in new_qas[:num]]

print(len(queries),len(revised_queries))

10 10


In [10]:
num = len(hard_qas)

queries = [item["question"] for item in hard_qas[:num]]

revised_queries = [item["question"] for item in hard_qas[:num]]

print(len(queries),len(revised_queries))

10 10


In [18]:
num = len(new_hard_qas)

queries = [item["question"] for item in new_hard_qas[:num]]

revised_queries = [item["question"] for item in new_hard_qas[:num]]

print(len(queries),len(revised_queries))

10 10


In [62]:
num = len(IDA_qas)

queries = [item["question"] for item in IDA_qas[:num]]

revised_queries = [item["question"] for item in IDA_qas[:num]]

print(len(queries),len(revised_queries))

7 7


In [6]:
if not llm:
    print("Skipping QueryManager test due to LLM initialization failure.")
else:
    print("--- Starting QueryManager Test ---")
    query_manager = QueryManager()

    # 更新类缓存
    print("Updating class name cache...")
    query_manager.update_class_name_cache(test_onto)
    # 打印部分缓存内容以确认
    if query_manager.class_name_cache:
         print(f"Cache content (first 10): {query_manager.class_name_cache[:10]}...")
    else:
         print("Class name cache is empty.")


    # 启动管理器
    print("Starting QueryManager...")
    query_manager.start()

    # 提交多个查询并收集futures
    futures = []
    print(f"\\nSubmitting {len(queries)} queries...")
    for i, query_text in enumerate(queries):
        print(f"Submitting query {i+1}: '{query_text[:80]}...'") # 打印部分查询文本
        future = query_manager.submit_query(query_text=query_text, query_context=query_context)
        futures.append((i+1, query_text, future))

    print("\nAll queries submitted.")

--- Starting QueryManager Test ---
Updating class name cache...
Class name cache updated with 351 classes.
Cache content (first 10): ['1,3,5-triethyl-2,4,6-trimethylamine', '1,3-diynyl', '1,4-triazole', '1:1_inclusion_complexes', '1H_NMR_spectroscopic_titration', '1H_NMR_spectroscopic_titrations', '1H_NMR_spectrum', '1_palmitoyl_2_oleoyl_sn_glycero_3_phosphocholine(POPC)', '23a', '25⊂(24)2']...
Starting QueryManager...
Dispatcher loop started on thread QueryDispatcherThread
Query Manager dispatcher started.
\nSubmitting 10 queries...
Submitting query 1: 'What description is provided for calix[4]pyrrole in the ontology?...'
Submitting query 2: 'What is the pKa value mentioned for methanesulfonic acid (MSA)?...'
Submitting query 3: 'What type of reaction is used to synthesize aryl-extended calix(4)pyrroles (AE-C...'
Submitting query 4: 'List some specific types (subclasses) of macrocyclic receptors mentioned in the ...'
Submitting query 5: 'According to the ontology, what can enhance the

http://www.test.org/chem_ontologies/meta/
http://www.test.org/chem_ontologies/classes/
http://www.test.org/chem_ontologies/object_properties/
http://www.test.org/chem_ontologies/data_properties/
http://www.test.org/chem_ontologies/meta/
http://www.test.org/chem_ontologies/classes/
http://www.test.org/chem_ontologies/object_properties/
http://www.test.org/chem_ontologies/data_properties/
http://www.test.org/chem_ontologies/meta/
http://www.test.org/chem_ontologies/classes/
http://www.test.org/chem_ontologies/object_properties/
http://www.test.org/chem_ontologies/data_properties/
http://www.test.org/chem_ontologies/meta/
http://www.test.org/chem_ontologies/classes/
http://www.test.org/chem_ontologies/object_properties/
http://www.test.org/chem_ontologies/data_properties/
{
  "results": [
    {
      "tool": "parse_class_definition",
      "params": {
        "class_names": [
          "calix(4)pyrrole"
        ]
      },
      "result": {
        "calix(4)pyrrole": {
          "basic_inf

In [7]:
if not llm:
    print("Skipping QueryManager test due to LLM initialization failure.")
else:    
    # 等待并获取所有结果
    print("Waiting for queries completion...")
    try:
        for i, query_text, future in futures:
            print(f"\nProcessing results for query {i}: '{query_text}'")
            # 等待合理的时间（根据需要调整）
            final_result_dict = future.result(timeout=120)
            print(f"Query {i} completed.")
            # 美观打印最终状态字典
            print(f"\n--- Final State Dictionary for Query {i} ---")
            # 使用default=str处理潜在的不可序列化对象，如本体引用
            print(json.dumps(final_result_dict, indent=2, default=str))
    except Exception as e:
        print(f"Error getting query result: {e}")
        if future.done() and future.exception():
             print(f"Future exception details: {future.exception()}")
    finally:
        # 停止管理器
        print("\nStopping QueryManager...")
        query_manager.stop()
        print("QueryManager stopped.")

    print("--- QueryManager Test Finished ---")


Waiting for queries completion...

Processing results for query 1: 'What description is provided for calix[4]pyrrole in the ontology?'
Query 1 completed.

--- Final State Dictionary for Query 1 ---
{
  "query": "What description is provided for calix[4]pyrrole in the ontology?",
  "source_ontology": "OntologySettings(base_iri='http://www.test.org/chem_ontologies/', ontology_file_name='backup-2.owl', directory_path='D:\\\\\\\\CursorProj\\\\\\\\Chem-Ontology-Constructor\\\\\\\\data\\\\ontology', closed_ontology_file_name='backup-2-closed.owl')",
  "query_type": "information_retrieval",
  "query_strategy": "tool_sequence",
  "originating_team": "test_notebook",
  "originating_stage": "manual_test",
  "available_classes": [
    "1,3,5-triethyl-2,4,6-trimethylamine",
    "1,3-diynyl",
    "1,4-triazole",
    "1:1_inclusion_complexes",
    "1H_NMR_spectroscopic_titration",
    "1H_NMR_spectroscopic_titrations",
    "1H_NMR_spectrum",
    "1_palmitoyl_2_oleoyl_sn_glycero_3_phosphocholine(POPC

In [19]:
queries = [queries[4]]
revised_queries = [revised_queries[4]]

In [30]:
# 定义回调函数处理Future结果并使用agent生成回答

def process_result_with_agent(result_dict, query_text):
    """
    Use an agent to process query results and generate a natural language response
    
    Args:
        result_dict: Query result dictionary
        query_text: Original query text
    
    Returns:
        str: Natural language response generated by the agent
    """
    # Extract query results information from the result
    if "formatted_results" in result_dict:
        query_results = result_dict["formatted_results"]
        print("-"*100)
        print(f"formatted_results found")
    elif "query_results" in result_dict:
        query_results = result_dict["query_results"]
        print("-"*100)
        print(f"query_results found")
    else:
        return f"I'm sorry, I couldn't find valid information about '{query_text}'."
    
    # Construct the prompt in English
    prompt = f"""
**Role:** You are an expert Chemistry Researcher.

**Task:** Provide a clear, accurate, and comprehensive answer to the user's question. You should leverage your own expert knowledge, **judiciously enhancing and verifying** it with **relevant and applicable information** selected from the 'Ontology query results'.

**User Question:**
{query_text}

**Information Source (Ontology Query Results for Enhancement & Verification):**
{query_results}

**Response Guidelines:**
* **Knowledge Integration:** Synthesize your broad chemical knowledge with **pertinent details** from the 'Information Source'.
* **Selective Use of Source:** Critically evaluate the 'Information Source'. **Incorporate specific details** (e.g., data points like pKa values, reaction types, precise definitions, specific examples) **only when they directly enhance the accuracy, specificity, or completeness of the answer to the user's question.** Do not feel obligated to include all provided information; prioritize relevance to the query.
* **Verification and Conflict:** Use the source to verify facts where appropriate. If there's a conflict between your general knowledge and the source, prioritize the source's specific data **if it is relevant to the question and appears accurate**, but use your expert judgment to omit information that seems erroneous or irrelevant to the user's query.
* **Synthesis:** Weave together your general knowledge and the selected source information into a coherent, well-structured response.
* **Clarity & Tone:** Use precise, professional chemical language. Aim for accessibility by briefly explaining potentially niche terms if needed.
* **Directness & Comprehensiveness:** Address all parts of the user's question directly and thoroughly, enriched by the appropriately selected information.
* **Source Attribution:** Do **not** mention "ontology" or refer to the 'Information Source' explicitly (e.g., avoid "according to the provided data..."). Present the integrated information as established chemical facts.
* **Knowledge Expansion:** Feel free to supplement the response with your own expert knowledge on topics that may be absent from the 'Ontology Query Results' but are relevant to providing a complete answer to the user's question.
* **Supramolecular Chemistry Style:** Tailor your response to appeal to supramolecular chemists by emphasizing host-guest interactions, non-covalent binding phenomena, molecular recognition principles, and structure-property relationships. Include relevant thermodynamic parameters, binding constants, and mechanistic insights where appropriate.

**Answer:**
"""
    
    # Generate response using LLM
    try:
        response = answer_llm.invoke(prompt)
        return response
    except Exception as e:
        return f"Error generating response: {e}"

def query_result_callback(future, query_idx, query_text):
    """Callback function to process Future results"""
    try:
        print(f"\nProcessing callback for query {query_idx}: '{query_text}'")
        
        # Get the future result
        result_dict = future.result(timeout=5)  # Small timeout to avoid indefinite waiting
        
        # Process the result using the agent
        answer = process_result_with_agent(result_dict, query_text)
        
        # Print the agent-generated answer
        print(f"\n--- Agent Answer for Query {query_idx} ---")
        print(answer)
        print("------------------------------")
        print(answer.content)
        
        return answer
    except Exception as e:
        print(f"Error processing result in callback: {e}")
        if future.exception():
            print(f"Future exception details: {future.exception()}")
        return None

# Test code using callback functions to process query results

if not llm:
    print("Skipping callback test due to LLM initialization failure.")
else:
    print("\n--- Starting Callback Function Test ---")
    
    # Re-create query manager if needed
    if 'query_manager' not in locals() or not hasattr(query_manager, 'is_running') or not query_manager.is_running():
        query_manager = QueryManager(max_workers=10)
        query_manager.update_all_caches(test_onto)
        query_manager.start()
    
    # 创建一个闭包函数来捕获回调返回的answer
    def create_answer_collector():
        # 在闭包中创建一个存储结果的字典
        answers = {}
        
        # 创建一个能捕获answer的回调函数
        def answer_collector(future, query_idx, query_text):
            try:
                result_dict = future.result(timeout=5)
                # 处理结果并获取answer
                answer = process_result_with_agent(result_dict, query_text)
                # 将answer存储在闭包的answers字典中
                answers[query_idx] = answer
                print(f"查询 {query_idx} 的答案已保存")
                return answer
            except Exception as e:
                print(f"处理结果时出错: {e}")
                return None
        
        # 返回回调函数和结果字典
        return answer_collector, answers

    # 创建回调函数和结果存储字典
    callback_collector, answers = create_answer_collector()

    # 提交查询并注册回调
    callback_futures = []
    for i, query_text in enumerate(queries):
        question = revised_queries[i]
        print(f"提交查询 {i+1}: '{query_text}'")
        future = query_manager.submit_query(query_text=query_text, query_context=query_context)
        
        # 使用functools.partial创建带参数的回调函数
        from functools import partial
        callback_func = partial(callback_collector, query_idx=i+1, query_text=question)
        
        # 注册回调函数
        future.add_done_callback(callback_func)
        callback_futures.append((i+1, query_text, future))
    
    # Wait for all Futures to complete (optional but ensures all callbacks execute)
    import concurrent.futures
    import time
    
    # Non-blocking check
    all_done = False
    wait_time = 0
    max_wait_time = 1200  # Maximum wait time
    check_interval = 5  # Check interval
    
    print("\nWaiting for callbacks to execute...")
    while not all_done and wait_time < max_wait_time:
        all_done = all(future[2].done() for future in callback_futures)
        if not all_done:
            print(f"Waited {wait_time} seconds, continuing to wait for callbacks...")
            time.sleep(check_interval)
            wait_time += check_interval
    
    if all_done:
        print("\nAll callbacks have completed!")
    else:
        print(f"\nTimeout waiting, some queries may not have completed. Waited {wait_time} seconds.")
    
    # Stop query manager
    print("\nStopping QueryManager...")
    query_manager.stop()
    print("QueryManager stopped.")
    
    print("--- Callback Function Test Finished ---")


--- Starting Callback Function Test ---
Class name cache updated with 13364 classes.
数据属性缓存更新完成，共 5859 个属性
对象属性缓存更新完成，共 4557 个属性
所有本体缓存更新完成
Dispatcher loop started on thread QueryDispatcherThread
Query Manager dispatcher started.
提交查询 1: 'Tell me about Quinine.'
提交查询 2: 'What is an Indicator Displacement Assay?'
提交查询 3: 'What techniques are used to analyze Quinine?'
提交查询 4: 'What are the components of an Indicator Displacement Assay?'
提交查询 5: 'Are there electrochemical sensors using Indicator Displacement Assay (IDA) to detect Quinine?'
提交查询 6: 'Which host molecules use host-guest recognition in electrochemical assays?'
提交查询 7: 'How stable and reproducible is the electrochemical sensor that uses an Indicator Displacement Assay (IDA) for detecting Quinine?'
提交查询 8: 'How is the electrochemical sensor that uses an Indicator Displacement Assay (IDA) for detecting Quinine verified?'
提交查询 9: 'In the electrochemical sensor that uses an Indicator Displacement Assay (IDA) for detecting Quinine

D:\\CursorProj\\Chem-Ontology-Constructor\autology_constructor\idea\query_team\ontology_tools.py:363: UserWarning: 在 'classes' 命名空间中找到 'electrochemical_signal' 但它不是 ThingClass。
  warnings.warn(f"在 'classes' 命名空间中找到 '{class_name}' 但它不是 ThingClass。")


Retry count: 1
{
  "results": [
    {
      "tool": "get_class_info",
      "params": {
        "class_names": [
          "indicator_displacement_assay(IDA)",
          "electrochemical_sensor",
          "quinine",
          "methylene_blue(MB)",
          "beta-cyclodextrin",
          "host-guest_chemistry",
          "binding_affinity",
          "inclusion_complex",
          "competitive_binding",
          "electrochemical_signal"
        ]
      },
      "result": {
        "indicator_displacement_assay(IDA)": {
          "name": "indicator_displacement_assay(IDA)",
          "information": [
            "Indicator displacement assay (IDA) is a method used for ultrasensitive fluorescence detection of LPA in aqueous media.",
            "Through IDA coupled with differential sensing, we achieved the ultrasensitive and specific detection of LPA."
          ]
        },
        "electrochemical_sensor": {
          "name": "electrochemical_sensor",
          "information": [
    

D:\\CursorProj\\Chem-Ontology-Constructor\autology_constructor\idea\query_team\ontology_tools.py:363: UserWarning: 在 'classes' 命名空间中找到 'indicator_molecule' 但它不是 ThingClass。
  warnings.warn(f"在 'classes' 命名空间中找到 '{class_name}' 但它不是 ThingClass。")
D:\\CursorProj\\Chem-Ontology-Constructor\autology_constructor\idea\query_team\ontology_tools.py:363: UserWarning: 在 'classes' 命名空间中找到 'electrochemical_signal' 但它不是 ThingClass。
  warnings.warn(f"在 'classes' 命名空间中找到 '{class_name}' 但它不是 ThingClass。")
D:\\CursorProj\\Chem-Ontology-Constructor\autology_constructor\idea\query_team\ontology_tools.py:363: UserWarning: 在 'classes' 命名空间中找到 'matrix_effects' 但它不是 ThingClass。
  warnings.warn(f"在 'classes' 命名空间中找到 '{class_name}' 但它不是 ThingClass。")
D:\\CursorProj\\Chem-Ontology-Constructor\autology_constructor\idea\query_team\ontology_tools.py:363: UserWarning: 在 'classes' 命名空间中找到 'statistical_validation' 但它不是 ThingClass。
  warnings.warn(f"在 'classes' 命名空间中找到 '{class_name}' 但它不是 ThingClass。")
D:\\CursorProj\\

Retry count: 1
Retry count: 1
In retry logic, Retry count: 1


...

Retry count: 4
{
  "results": [
    {
      "tool": "get_class_info",
      "params": {
        "class_names": [
          "electrochemical_sensor",
          "indicator_displacement_assay(IDA)",
          "quinine",
          "receptor",
          "indicator",
          "calibration_curve",
          "selectivity",
          "interference",
          "reproducibility",
          "stability",
          "real_sample_analysis",
          "control_experiment"
        ]
      },
      "result": {
        "electrochemical_sensor": {
          "name": "electrochemical_sensor",
          "information": [
            "An electrochemical sensor employing host-guest interactions of Q(8) was applied to the determination of tryptophan in real samples.",
            "Electrochemical sensors belong to the established methods of nucleotide detection.",
            "Nanocomposite based electrochemical sensor properties of polyaniline decorated with silver nanoparticles were utilized for the determinat

D:\\CursorProj\\Chem-Ontology-Constructor\autology_constructor\idea\query_team\ontology_tools.py:363: UserWarning: 在 'classes' 命名空间中找到 'surface_immobilization' 但它不是 ThingClass。
  warnings.warn(f"在 'classes' 命名空间中找到 '{class_name}' 但它不是 ThingClass。")


Retry count: 4
{
  "results": [
    {
      "tool": "get_class_info",
      "params": {
        "class_names": [
          "graphene"
        ]
      },
      "result": {
        "graphene": {
          "name": "graphene",
          "information": []
        }
      }
    },
    {
      "tool": "get_class_properties",
      "params": {
        "class_names": [
          "graphene"
        ]
      },
      "result": {
        "graphene": {
          "electrical_conductivity": {
            "restrictions": [
              {
                "type": "VALUE",
                "value": "superior",
                "raw_value": "superior"
              },
              {
                "type": "VALUE",
                "value": "superior",
                "raw_value": "superior"
              }
            ],
            "descriptions": []
          },
          "electron_transfer_property": {
            "restrictions": [
              {
                "type": "VALUE",
                "value"

D:\\CursorProj\\Chem-Ontology-Constructor\autology_constructor\idea\query_team\ontology_tools.py:363: UserWarning: 在 'classes' 命名空间中找到 'signal_transduction' 但它不是 ThingClass。
  warnings.warn(f"在 'classes' 命名空间中找到 '{class_name}' 但它不是 ThingClass。")
D:\\CursorProj\\Chem-Ontology-Constructor\autology_constructor\idea\query_team\ontology_tools.py:363: UserWarning: 在 'classes' 命名空间中找到 'analytical_detection_and_quantification' 但它不是 ThingClass。
  warnings.warn(f"在 'classes' 命名空间中找到 '{class_name}' 但它不是 ThingClass。")


Retry count: 4
{
  "results": [
    {
      "tool": "parse_class_definition",
      "params": {
        "class_names": [
          "indicator_displacement_assay",
          "receptor",
          "indicator",
          "analyte",
          "competitive_binding",
          "signal_transduction",
          "binding_affinity",
          "supramolecular_chemistry",
          "analytical_detection_and_quantification"
        ]
      },
      "result": {
        "indicator_displacement_assay": {
          "basic_info": {
            "name": "indicator_displacement_assay",
            "information": [
              "Post-translational modifications in proteins are mainly detected using indicator displacement assays.",
              "Indicator displacement assays are analytical techniques in which a host molecule binds to an indicator dye, and the displacement of the dye by a guest molecule is used to detect the presence of the guest.",
              "Indicator displacement assay, coupled with 

In [31]:
future_list = [future[2].result() for future in callback_futures]
future_list

[{'query': 'Tell me about Quinine.',
  'source_ontology': OntologySettings(base_iri='http://www.test.org/chem_ontologies/', ontology_file_name='final.owl', directory_path='D:\\\\CursorProj\\\\Chem-Ontology-Constructor\\\\data\\ontology', closed_ontology_file_name='IDA-closed.owl'),
  'query_type': 'information_retrieval',
  'query_strategy': 'tool_sequence',
  'originating_team': 'test_notebook',
  'originating_stage': 'manual_test',
  'available_classes': ['(2)pseudorotaxane',
   '(3)pseudorotaxane',
   '(4)(cl)3',
   '(4)(pf6)3',
   '(M + H)+',
   '(Nb6Cl12(H2O)6)@(gamma-CD)2)2+',
   '(Ta6Br12(H2O)6)@(gamma-CD)2)Br2.14H2O',
   '(bmim)',
   '(bmim)(octs)',
   '(cl)-',
   '(cu4)(cl)·5',
   '(ethylamino)carbonyl-2-pyridinecarboxylic_acid',
   '(h4)(pf6)2',
   '(nb6br12)n+',
   '(nb6cl12)n+',
   '(nb6i12)n+',
   '(octs)',
   '(pf6)-',
   '(ta6br12)n+',
   '(ta6cl12)n+',
   '(ta6i12)n+',
   '1,1-butane(1,4-diyl)bis(2-aminopyridine)_bromide(DPAD)',
   '1,1-ferrocenedicarboxylic_acid',
   '

In [32]:
future_list[2]


{'query': 'What techniques are used to analyze Quinine?',
 'source_ontology': OntologySettings(base_iri='http://www.test.org/chem_ontologies/', ontology_file_name='final.owl', directory_path='D:\\\\CursorProj\\\\Chem-Ontology-Constructor\\\\data\\ontology', closed_ontology_file_name='IDA-closed.owl'),
 'query_type': 'information_retrieval',
 'query_strategy': 'tool_sequence',
 'originating_team': 'test_notebook',
 'originating_stage': 'manual_test',
 'available_classes': ['(2)pseudorotaxane',
  '(3)pseudorotaxane',
  '(4)(cl)3',
  '(4)(pf6)3',
  '(M + H)+',
  '(Nb6Cl12(H2O)6)@(gamma-CD)2)2+',
  '(Ta6Br12(H2O)6)@(gamma-CD)2)Br2.14H2O',
  '(bmim)',
  '(bmim)(octs)',
  '(cl)-',
  '(cu4)(cl)·5',
  '(ethylamino)carbonyl-2-pyridinecarboxylic_acid',
  '(h4)(pf6)2',
  '(nb6br12)n+',
  '(nb6cl12)n+',
  '(nb6i12)n+',
  '(octs)',
  '(pf6)-',
  '(ta6br12)n+',
  '(ta6cl12)n+',
  '(ta6i12)n+',
  '1,1-butane(1,4-diyl)bis(2-aminopyridine)_bromide(DPAD)',
  '1,1-ferrocenedicarboxylic_acid',
  '1,2-bis(

In [33]:
future_list = [future[2].result() for future in callback_futures]
# [future["formatted_results"] for future in future_list]
future_list[0]["formatted_results"]
for i, future_item in enumerate(future_list):
    print(f"Processing item {i}:")
    try:
        # 尝试访问，看看哪个会出错
        formatted_res = future_item["formatted_results"]
        print(f"  Type of future_item: {type(future_item)}")
        print(f"  Keys in future_item: {future_item.keys() if hasattr(future_item, 'keys') else 'N/A'}")
        print(f"  Type of formatted_results: {type(formatted_res)}")
        print(f"  Value of formatted_results: {formatted_res}")
    except Exception as e:
        print(f"  Error accessing 'formatted_results' for item {i}: {e}")
        print(f"  Type of future_item that caused error: {type(future_item)}")
        if hasattr(future_item, 'keys'):
            print(f"  Keys in future_item: {future_item.keys()}")
        else:
            print(f"  future_item does not have 'keys' attribute.")
        # 如果需要，可以打印整个 problematic future_item
        # print(f"  Problematic future_item: {future_item}")

Processing item 0:
  Type of future_item: <class 'langgraph.pregel.io.AddableValuesDict'>
  Keys in future_item: dict_keys(['query', 'source_ontology', 'query_type', 'query_strategy', 'originating_team', 'originating_stage', 'available_classes', 'available_data_properties', 'available_object_properties', 'query_results', 'normalized_query', 'execution_plan', 'validation_report', 'status', 'stage', 'previous_stage', 'retry_count', 'hypothetical_document', 'formatted_results', 'messages'])
  Type of formatted_results: <class 'dict'>
  Value of formatted_results: {'summary': 'Quinine is a naturally occurring alkaloid primarily obtained from the bark of the cinchona tree, known for its use as a pharmacological agent, especially in the treatment of malaria. It has a complex chemical structure and is also used in flavoring, notably in tonic water.', 'key_points': ['Quinine is classified as an alkaloid, a group of naturally occurring organic compounds containing nitrogen.', 'It is primarily e

In [34]:
answers

{4: AIMessage(content="An Indicator Displacement Assay (IDA) comprises four primary components, each playing a crucial role in the detection mechanism:\n\n1. **Host (Receptor):**  \n   This is a molecular entity—often a macrocyclic compound, a metal-ligand complex, or a supramolecular assembly—that exhibits selective binding affinity for the indicator molecule. The host's binding site is designed to recognize specific features of the indicator, forming a stable host-guest complex through non-covalent interactions such as hydrogen bonding, π-π stacking, electrostatic interactions, or metal coordination. The thermodynamic stability of this complex (reflected in the binding constant, K_a) ensures a measurable baseline signal.\n\n2. **Indicator (Reporter):**  \n   The indicator is a molecule capable of producing a detectable signal—commonly a colorimetric or fluorescent response—when bound to the host. Its signal change upon binding is well-characterized, with properties such as pKa, absor

In [35]:
from langchain_core.messages import AIMessage
for idx, answer in answers.items():
        if isinstance(answer, AIMessage):
                q = queries[idx-1]
                print(f"{idx}.查询 {q} 的最终答案: {answer.content}\n")
        else:
                print("error")

4.查询 What are the components of an Indicator Displacement Assay? 的最终答案: An Indicator Displacement Assay (IDA) comprises four primary components, each playing a crucial role in the detection mechanism:

1. **Host (Receptor):**  
   This is a molecular entity—often a macrocyclic compound, a metal-ligand complex, or a supramolecular assembly—that exhibits selective binding affinity for the indicator molecule. The host's binding site is designed to recognize specific features of the indicator, forming a stable host-guest complex through non-covalent interactions such as hydrogen bonding, π-π stacking, electrostatic interactions, or metal coordination. The thermodynamic stability of this complex (reflected in the binding constant, K_a) ensures a measurable baseline signal.

2. **Indicator (Reporter):**  
   The indicator is a molecule capable of producing a detectable signal—commonly a colorimetric or fluorescent response—when bound to the host. Its signal change upon binding is well-charac

In [46]:
res_list = []
for i, ques in enumerate(revised_queries):
    response = answer_llm.invoke(ques)
    res_list.append(response)
    print(f"完成 {i+1} 个回答")


完成 1 个回答
完成 2 个回答
完成 3 个回答
完成 4 个回答
完成 5 个回答
完成 6 个回答
完成 7 个回答
完成 8 个回答
完成 9 个回答
完成 10 个回答


In [47]:
res_list

[AIMessage(content='A **cryptand** is a type of synthetic, cage-like molecule designed to bind specific ions or molecules within its structure. The term comes from the Greek "kryptos" (hidden) and "and" (from "andros," meaning man, but here used as a suffix for "container"). Cryptands are a class of **macrocyclic ligands**—they are three-dimensional analogues of crown ethers.\n\n### Structure\n- Cryptands are typically composed of several interconnected polyether chains, forming a three-dimensional cavity or "crypt" that can encapsulate a guest ion (often a metal cation).\n- The most common cryptands are based on nitrogen and oxygen atoms as donor sites, such as [2.2.2]cryptand, which has three bridges, each with two oxygen atoms, connecting three nitrogen atoms.\n\n### Function\n- **Host-guest chemistry:** Cryptands are excellent at selectively binding (chelating) metal ions, especially alkali and alkaline earth metals, due to their preorganized cavities.\n- **Complexation:** The enca

In [48]:
for i, res in enumerate(res_list):
    q = queries[i]
    print(f"{i+1}.查询 {q} 的答案是：{res.content}\n")

1.查询 What is a cryptand? 的答案是：A **cryptand** is a type of synthetic, cage-like molecule designed to bind specific ions or molecules within its structure. The term comes from the Greek "kryptos" (hidden) and "and" (from "andros," meaning man, but here used as a suffix for "container"). Cryptands are a class of **macrocyclic ligands**—they are three-dimensional analogues of crown ethers.

### Structure
- Cryptands are typically composed of several interconnected polyether chains, forming a three-dimensional cavity or "crypt" that can encapsulate a guest ion (often a metal cation).
- The most common cryptands are based on nitrogen and oxygen atoms as donor sites, such as [2.2.2]cryptand, which has three bridges, each with two oxygen atoms, connecting three nitrogen atoms.

### Function
- **Host-guest chemistry:** Cryptands are excellent at selectively binding (chelating) metal ions, especially alkali and alkaline earth metals, due to their preorganized cavities.
- **Complexation:** The en

**Note on Streaming with QueryManager:**

The standard `QueryManager.submit_query()` returns a `Future` that resolves to the *final* state of the LangGraph execution. It doesn't inherently provide access to the intermediate states generated by each node.

To observe the step-by-step execution and intermediate state changes, you would typically need to interact directly with the LangGraph instance using its `stream()` method, as demonstrated in Part 2 below. Modifying the `QueryManager` to expose this stream would require significant changes to its asynchronous task handling and result reporting.

## Part 2: Direct Execution via Graph Stream

In [15]:
if not llm:
    print("Skipping Direct Graph Stream test due to LLM initialization failure.")
else:
    print("\n--- Starting Direct Graph Stream Test ---")

    # 1. Create graph instance
    print("Creating graph instance...")
    graph = create_query_graph()
    print("Graph instance created.")

    # 2. Manually create initial state dictionary
    print("Creating initial state...")
    # Use the same query as Part 1 for comparison
    # query_text_stream = "What proteins does DrugA bind to?"

    for query_text_stream in [queries[4]]:

        try:
            # Ensure we get a list of strings
            available_classes_stream = sorted([cls.name for cls in test_onto.classes() if isinstance(cls, ThingClass)])
            available_data_props_stream = sorted([dp.name for dp in test_onto.data_properties() if isinstance(dp, DataPropertyClass)])
            available_object_props_stream = sorted([op.name for op in test_onto.object_properties() if isinstance(op, ObjectPropertyClass)])
        except Exception as e:
            print(f"Error getting class names: {e}")
            available_classes_stream = []

        initial_state = {
            "query": query_text_stream,
            "source_ontology": test_ontology_settings, # Pass the actual ontology object
            "available_classes": available_classes_stream,
            "available_data_properties": available_data_props_stream,
            "available_object_properties": available_object_props_stream,
            "query_type": "information_retrieval",
            "query_strategy": None,
            "originating_team": "test_notebook_stream",
            "originating_stage": "manual_stream_test",
            "query_results": {},
            "normalized_query": None,
            "execution_plan": None,
            "validation_report": None,
            "sparql_query": None,
            "status": "initialized",
            "stage": "initialized",
            "previous_stage": None,
            "error": None,
            "messages": [] # LangGraph expects messages field
        }
        print("Initial state prepared.")
        # print(json.dumps(initial_state, indent=2, default=str)) # Optionally print initial state (ontology won't serialize well)

        # 3. Execute and iterate stream
        print("\n--- Streaming Graph Execution --- ")
        try:
            stream_counter = 0
            # Use stream method to get intermediate steps
            for chunk in graph.stream(initial_state, config={"recursion_limit": 50}):
                stream_counter += 1
                print(f"\n--- Chunk {stream_counter} --- ")
                # Chunks are dictionaries where keys are node names that just ran
                # and values are the outputs (state updates) returned by that node
                # Use default=str to handle potential non-serializable objects in the state
                print(json.dumps(chunk, indent=2, default=str))
                print("-" * 30)
            print("\n--- Graph Stream Finished --- ")
        except Exception as e:
            print(f"\nError during graph stream: {e}")
            import traceback
            traceback.print_exc() # Print full traceback for stream errors
        
        print(f"query:{query_text_stream} has been finished.")

    print("--- Direct Graph Stream Test Finished ---")


--- Starting Direct Graph Stream Test ---
Creating graph instance...
Graph instance created.
Creating initial state...
Initial state prepared.

--- Streaming Graph Execution --- 
Retry count: 1

--- Chunk 1 --- 
{
  "normalize": {
    "normalized_query": "intent='find information' relevant_entities=['electrochemical_sensor', 'indicator_displacement_assay(IDA)', 'quinine'] relevant_properties=[] filters=None query_type_suggestion='fact-finding'",
    "status": "parsing_complete",
    "stage": "normalized",
    "previous_stage": "initialized",
    "retry_count": 1,
    "messages": [
      "content='Query normalized: Are there electrochemical sensors using Indicator Displacement Assay (IDA) to detect Quinine?' additional_kwargs={} response_metadata={} id='767013f5-1244-430d-82ad-d8c7b8f57278'"
    ]
  }
}
------------------------------
Retry count: 1

--- Chunk 2 --- 
{
  "strategy": {
    "query_strategy": "tool_sequence",
    "status": "strategy_determined",
    "stage": "strategy",
  

In [17]:
print(graph["normalized_query"])

TypeError: 'CompiledStateGraph' object is not subscriptable

# QueryManager 检查


In [7]:
import threading
import traceback
import sys

def check_threads():
    """检查当前进程中的活跃线程"""
    print(f"当前活跃线程数: {threading.active_count()}")
    
    print("\n当前活跃线程:")
    for t in threading.enumerate():
        print(f"- {t.name} (daemon: {t.daemon}, 活动: {t.is_alive()})")
    
    print("\n线程调用栈:")
    query_manager_threads = []
    for thread_id, frame in sys._current_frames().items():
        thread_name = "Unknown"
        for t in threading.enumerate():
            if t.ident == thread_id:
                thread_name = t.name
                break
        
        # 检查是否是QueryManager相关线程
        is_query_thread = False
        stack_trace = traceback.extract_stack(frame)
        for filename, _, _, _ in stack_trace:
            if "query_manager" in filename or "ThreadPool" in filename:
                is_query_thread = True
                query_manager_threads.append(thread_name)
                break
        
        print(f"线程ID: {thread_id}, 名称: {thread_name}{' (QueryManager相关)' if is_query_thread else ''}")
        for filename, lineno, name, line in stack_trace[-10:]:  # 只显示最近10个调用
            print(f"  文件: {filename.split('/')[-1]}, 行: {lineno}, 函数: {name}")
            if line:
                print(f"    代码: {line}")
        print("")
    
    if query_manager_threads:
        print(f"\n发现 {len(query_manager_threads)} 个QueryManager相关线程: {', '.join(query_manager_threads)}")
    else:
        print("\n未发现QueryManager相关线程")

# 执行检查
check_threads()

当前活跃线程数: 6

当前活跃线程:
- MainThread (daemon: False, 活动: True)
- IOPub (daemon: True, 活动: True)
- Heartbeat (daemon: True, 活动: True)
- Control (daemon: True, 活动: True)
- IPythonHistorySavingThread (daemon: True, 活动: True)
- Thread-1 (daemon: True, 活动: True)

线程调用栈:
线程ID: 13768, 名称: Thread-1
  文件: d:\AnacondaEnPs\envs\OntologyConstruction\Lib\threading.py, 行: 1012, 函数: _bootstrap
    代码: self._bootstrap_inner()
  文件: d:\AnacondaEnPs\envs\OntologyConstruction\Lib\threading.py, 行: 1041, 函数: _bootstrap_inner
    代码: self.run()
  文件: d:\AnacondaEnPs\envs\OntologyConstruction\Lib\site-packages\ipykernel\parentpoller.py, 行: 93, 函数: run
    代码: result = ctypes.windll.kernel32.WaitForMultipleObjects(  # type:ignore[attr-defined]

线程ID: 33716, 名称: IPythonHistorySavingThread
  文件: d:\AnacondaEnPs\envs\OntologyConstruction\Lib\threading.py, 行: 1012, 函数: _bootstrap
    代码: self._bootstrap_inner()
  文件: d:\AnacondaEnPs\envs\OntologyConstruction\Lib\threading.py, 行: 1041, 函数: _bootstrap_inner
    代码: sel

In [8]:
# 查看缓存内容（如果有缓存的查询）
cache_content = query_manager.query_queue_manager.cache.cache
print(f"缓存中的查询数量: {len(cache_content)}")

# 查看缓存的时间戳信息
timestamps = query_manager.query_queue_manager.cache.timestamps
if timestamps:
    print("\n缓存时间戳:")
    for key, timestamp in timestamps.items():
        print(f"查询: {key[:50]}... - 时间: {timestamp}")
        # 计算剩余有效时间
        ttl = query_manager.query_queue_manager.cache.ttl  # 默认3600秒（1小时）
        from datetime import datetime, timedelta
        remaining = timestamp + timedelta(seconds=ttl) - datetime.now()
        print(f"  剩余有效时间: {remaining}")

# 如果需要手动清除缓存
# query_manager.query_queue_manager.cache.clear()
# print("缓存已清除")

缓存中的查询数量: 0
